# LeetCode 3802 — Number of Ways to Paint Sheets bằng Python

- Link: https://leetcode.com/problems/number-of-ways-to-paint-sheets/
- Độ khó: **Hard** · 🔒 **Premium**
- Chủ đề: Đếm tổ hợp, đếm cặp có tổng vượt ngưỡng, sắp xếp + tìm nhị phân / hai con trỏ, tổng tiền tố, modulo

> **Ghi chú Premium:** đề bài viết lại theo hiểu biết về bài (dựa trên bản mô tả công khai của doocs). Skeleton dưới đây **suy luận** (doocs chưa có code), cần đối chiếu tên phương thức nếu có tài khoản Premium.

Skeleton (suy luận):

```python
class Solution:
    def numberOfWays(self, n: int, limit: List[int]) -> int:
```

## 1. Mục tiêu học

1. Mô hình hóa: một cách sơn = **cặp màu có thứ tự** `(i, j)`, `i ≠ j`, và một **điểm cắt** `x` (số tờ của màu đầu).
2. Đếm số `x` hợp lệ cho một cặp bằng **giao hai khoảng** ⇒ công thức `max(0, A_i + A_j − (n − 1))` với `A = min(limit, n − 1)`.
3. Tính `Σ_{i ≠ j} max(0, A_i + A_j − c)` cho `m = 10^5` màu trong `O(m log m)` bằng **sắp xếp + tìm nhị phân + tổng tiền tố** hoặc **hai con trỏ**.
4. Cẩn thận với modulo và tràn số (tổng có thể tới `~10^{19}`).
5. Kiểm thử với vét cạn.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n` tờ giấy xếp thành hàng và `m` màu, màu `i` sơn được **tối đa `limit[i]`** tờ. Phải sơn mọi tờ sao cho:

- dùng **đúng hai** màu khác nhau;
- mỗi màu phủ **một đoạn liên tiếp** (vậy hàng có dạng: `x` tờ màu thứ nhất rồi `n − x` tờ màu thứ hai, `1 ≤ x ≤ n − 1`);
- số tờ của mỗi màu không vượt giới hạn.

Đếm số cách (hai cách khác nhau nếu có một tờ khác màu), modulo `10^9 + 7`.

**Ví dụ 1:** `n = 4, limit = [3,1,2]` → `6`.

**Ví dụ 2:** `n = 3, limit = [1,2]` → `2`.

**Ví dụ 3:** `n = 3, limit = [2,2]` → `4`.

**Ràng buộc:** `2 ≤ n ≤ 10^9`, `2 ≤ m ≤ 10^5`, `1 ≤ limit[i] ≤ 10^9`.

In [1]:
from typing import List
from bisect import bisect_left
import random, time

MOD = 10**9 + 7
examples = [(4, [3, 1, 2], 6), (3, [1, 2], 2), (3, [2, 2], 4)]
for n, lim, exp in examples:
    print("n =", n, "limit =", lim, "-> kỳ vọng", exp)

n = 4 limit = [3, 1, 2] -> kỳ vọng 6
n = 3 limit = [1, 2] -> kỳ vọng 2
n = 3 limit = [2, 2] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Đếm số nguyên trong giao hai khoảng

Số nguyên trong `[lo, hi]` là `max(0, hi − lo + 1)`. Điều kiện `x ≤ a` và `n − x ≤ b` ⇔ `x ∈ [n − b, a]`; giao thêm với `[1, n − 1]`.

In [2]:
def count_x(n, a, b):
    lo, hi = max(1, n - b), min(a, n - 1)
    return max(0, hi - lo + 1)

print(count_x(4, 3, 1), count_x(4, 3, 2), count_x(4, 1, 3), count_x(3, 2, 2))

1 2 1 2


### 3.2. Cắt giới hạn về `n − 1`

Một màu không bao giờ sơn quá `n − 1` tờ (màu kia cần ít nhất 1 tờ). Đặt `A_i = min(limit[i], n − 1)`. Khi đó `n − A_j ≥ 1` và `A_i ≤ n − 1`, nên số `x` hợp lệ là `max(0, A_i − (n − A_j) + 1) = max(0, A_i + A_j − (n − 1))`.

In [3]:
n = 4
for a, b in [(3, 1), (3, 2), (10, 10), (1, 1)]:
    A, B = min(a, n - 1), min(b, n - 1)
    print((a, b), "->", count_x(n, a, b), "| công thức:", max(0, A + B - (n - 1)))

(3, 1) -> 1 | công thức: 1
(3, 2) -> 2 | công thức: 2
(10, 10) -> 3 | công thức: 3
(1, 1) -> 0 | công thức: 0


### 3.3. Đếm cặp có tổng vượt ngưỡng trên mảng đã sắp

Với mảng tăng `a` và ngưỡng `c`: số `j > i` có `a_i + a_j > c` là `n − max(i + 1, bisect_left(a, c − a_i + 1))`. Kèm **tổng tiền tố** để cộng nhanh `Σ a_j` trên đoạn.

In [4]:
a = sorted([1, 4, 2, 7, 5]); c = 7
from itertools import accumulate
pre = list(accumulate(a, initial=0))
tot = 0
for i, x in enumerate(a):
    p = max(i + 1, bisect_left(a, c - x + 1))
    tot += (len(a) - p) * (x - c) + (pre[-1] - pre[p])
print(a, "Σ_{i<j} max(0, a_i + a_j - c) =", tot, "| trực tiếp:", sum(max(0, a[i] + a[j] - c) for i in range(5) for j in range(i + 1, 5)))

[1, 2, 4, 5, 7] Σ_{i<j} max(0, a_i + a_j - c) = 14 | trực tiếp: 14


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `n = 4`, `limit = [3, 1, 2]` ⇒ `A = [3, 1, 2]` (đều ≤ 3 = n − 1), `c = n − 1 = 3`.

| cặp (i, j) | A_i + A_j − 3 | số cách |
|---|---|---|
| (0, 1), (1, 0) | 3 + 1 − 3 = 1 | 1 mỗi chiều |
| (0, 2), (2, 0) | 3 + 2 − 3 = 2 | 2 mỗi chiều |
| (1, 2), (2, 1) | 1 + 2 − 3 = 0 | 0 |

Tổng `2·(1 + 2 + 0) = 6` ✔. Nhận xét: số cách của `(i, j)` và `(j, i)` **bằng nhau** (công thức đối xứng) ⇒ đáp án = `2 × Σ_{i<j} max(0, A_i + A_j − c)`.

## 5. Cách 1 — Brute Force: thử mọi cặp màu và mọi điểm cắt

`O(m² · n)` — chỉ cho dữ liệu nhỏ.

In [5]:
def brute(n, limit):
    m = len(limit); cnt = 0
    for i in range(m):
        for j in range(m):
            if i == j: continue
            for x in range(1, n):
                if x <= limit[i] and n - x <= limit[j]:
                    cnt += 1
    return cnt % MOD

for n, lim, exp in examples:
    print(brute(n, lim), "kỳ vọng", exp)

6 kỳ vọng 6
2 kỳ vọng 2
4 kỳ vọng 4


**Nút thắt:** duyệt `x` tới `10^9` và cặp màu tới `10^{10}`. Bỏ duyệt `x` bằng công thức (mục 3.2), bỏ duyệt cặp bằng sắp xếp + tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án** = `2 · Σ_{i<j} max(0, A_i + A_j − c)` với `A_i = min(limit[i], n − 1)`, `c = n − 1`.

**Tách tổng:** sắp `A` tăng dần. Với mỗi `i`, các `j > i` thỏa `A_i + A_j > c` tạo thành **hậu tố** `[p, m)` (vì `A` tăng). Đóng góp:

`Σ_{j ∈ [p, m)} (A_i + A_j − c) = (m − p)(A_i − c) + Σ_{j ∈ [p, m)} A_j`.

`p` tìm bằng tìm nhị phân (hoặc hai con trỏ, vì khi `A_i` tăng thì `p` giảm), tổng hậu tố bằng tiền tố.

Toàn bộ tính bằng số nguyên Python rồi lấy modulo ở cuối (hoặc từng bước).

## 7. Các cách tiếp theo

- **Cách 2:** duyệt mọi cặp `(i, j)` với công thức đóng — `O(m²)`.
- **Cách 3:** sắp xếp + tìm nhị phân + tổng tiền tố — `O(m log m)` (lời giải nộp chính).
- **Cách 4:** sắp xếp + **hai con trỏ** (không cần `bisect`) — `O(m log m)` cho sắp xếp, `O(m)` phần đếm.

### 7.1. Cách 2 — Mọi cặp với công thức

In [6]:
def pairs_quadratic(n, limit):
    c = n - 1
    A = [min(x, c) for x in limit]
    m = len(A); tot = 0
    for i in range(m):
        for j in range(i + 1, m):
            tot += max(0, A[i] + A[j] - c)
    return 2 * tot % MOD

for n, lim, exp in examples:
    print(pairs_quadratic(n, lim), "kỳ vọng", exp)

6 kỳ vọng 6
2 kỳ vọng 2
4 kỳ vọng 4


**Độ phức tạp:** `O(m²) = 10^{10}` với `m = 10^5` — quá chậm.

### 7.2. Cách 3 — Sắp xếp + tìm nhị phân + tiền tố (lời giải chính)

In [7]:
def sort_bisect(n, limit):
    c = n - 1
    A = sorted(min(x, c) for x in limit)
    m = len(A)
    pre = [0] * (m + 1)
    for i, x in enumerate(A): pre[i + 1] = pre[i] + x
    tot = 0
    for i, x in enumerate(A):
        p = max(i + 1, bisect_left(A, c - x + 1))       # j >= p  <=>  A_j >= c - A_i + 1
        if p < m:
            tot += (m - p) * (x - c) + (pre[m] - pre[p])
    return 2 * tot % MOD

for n, lim, exp in examples:
    print(sort_bisect(n, lim), "kỳ vọng", exp)

6 kỳ vọng 6
2 kỳ vọng 2
4 kỳ vọng 4


**Độ phức tạp Cách 3:** `O(m log m)`; bộ nhớ `O(m)`.

### 7.3. Cách 4 — Hai con trỏ

Duyệt `i` từ trái; con trỏ `p` (đầu hậu tố hợp lệ) chỉ **giảm** khi `A_i` tăng. Bắt đầu `p = m`; với mỗi `i`, trong khi `p − 1 > i` và `A_i + A_{p−1} > c` thì `p −= 1`. Lưu ý phải giữ `p ≥ i + 1`.

In [8]:
def two_pointers(n, limit):
    c = n - 1
    A = sorted(min(x, c) for x in limit)
    m = len(A)
    suf = [0] * (m + 1)
    for i in range(m - 1, -1, -1): suf[i] = suf[i + 1] + A[i]
    tot = 0; p = m
    for i, x in enumerate(A):
        if p <= i: p = i + 1
        while p - 1 > i and A[p - 1] + x > c:
            p -= 1
        if p < m:
            tot += (m - p) * (x - c) + suf[p]
    return 2 * tot % MOD

for n, lim, exp in examples:
    print(two_pointers(n, lim), "kỳ vọng", exp)

6 kỳ vọng 6
2 kỳ vọng 2
4 kỳ vọng 4


In [9]:
class SolutionTwoPointers:
    def numberOfWays(self, n: int, limit: List[int]) -> int:
        return two_pointers(n, limit)

print([SolutionTwoPointers().numberOfWays(n, l) for n, l, _ in examples])

[6, 2, 4]


**Độ phức tạp Cách 4:** sắp xếp `O(m log m)` + quét `O(m)`.

*Vì sao `p` không cần tăng lại?* Khi `i` tăng, `A_i` không giảm ⇒ ngưỡng `c − A_i + 1` không tăng ⇒ chỉ số đầu hậu tố hợp lệ không tăng — trừ ràng buộc `p ≥ i + 1`, xử lý bằng `if p <= i: p = i + 1`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`n = 4`, `A = [1, 2, 3]` sau khi sắp, `c = 3`).

In [10]:
def trace(n, limit):
    c = n - 1; A = sorted(min(x, c) for x in limit); m = len(A)
    pre = [0] * (m + 1)
    for i, x in enumerate(A): pre[i + 1] = pre[i] + x
    print("A đã sắp =", A, "| c =", c)
    tot = 0
    for i, x in enumerate(A):
        p = max(i + 1, bisect_left(A, c - x + 1))
        add = (m - p) * (x - c) + (pre[m] - pre[p]) if p < m else 0
        tot += add
        print(f"i={i} A_i={x}: cần A_j >= {c - x + 1}, hậu tố bắt đầu p={p} -> cộng {add}")
    print("Σ_{i<j} =", tot, "-> đáp án = 2 ×", tot, "=", 2 * tot)

trace(4, [3, 1, 2])

A đã sắp = [1, 2, 3] | c = 3
i=0 A_i=1: cần A_j >= 3, hậu tố bắt đầu p=2 -> cộng 1
i=1 A_i=2: cần A_j >= 2, hậu tố bắt đầu p=2 -> cộng 2
i=2 A_i=3: cần A_j >= 1, hậu tố bắt đầu p=3 -> cộng 0
Σ_{i<j} = 3 -> đáp án = 2 × 3 = 6


In [11]:
# Trace với giới hạn vượt n - 1 (bị cắt)
trace(5, [10, 1, 3, 4])
print("vét cạn:", brute(5, [10, 1, 3, 4]))

A đã sắp = [1, 3, 4, 4] | c = 4
i=0 A_i=1: cần A_j >= 4, hậu tố bắt đầu p=2 -> cộng 2
i=1 A_i=3: cần A_j >= 2, hậu tố bắt đầu p=2 -> cộng 6
i=2 A_i=4: cần A_j >= 1, hậu tố bắt đầu p=3 -> cộng 4
i=3 A_i=4: cần A_j >= 1, hậu tố bắt đầu p=4 -> cộng 0
Σ_{i<j} = 12 -> đáp án = 2 × 12 = 24
vét cạn: 24


## 9. Code nộp LeetCode

In [12]:
from bisect import bisect_left

class Solution:
    def numberOfWays(self, n: int, limit: List[int]) -> int:
        MOD = 10**9 + 7
        c = n - 1
        A = sorted(min(x, c) for x in limit)
        m = len(A)
        pre = [0] * (m + 1)
        for i, x in enumerate(A):
            pre[i + 1] = pre[i] + x
        tot = 0
        for i, x in enumerate(A):
            p = max(i + 1, bisect_left(A, c - x + 1))
            if p < m:
                tot += (m - p) * (x - c) + pre[m] - pre[p]
        return 2 * tot % MOD

sol = Solution()
for n, lim, exp in examples:
    got = sol.numberOfWays(n, lim)
    print((n, lim), "->", got, "✔" if got == exp else "✘")

(4, [3, 1, 2]) -> 6 ✔
(3, [1, 2]) -> 2 ✔
(3, [2, 2]) -> 4 ✔


## 10. Vì sao thuật toán đúng

1. **Song ánh:** mỗi cách sơn hợp lệ ↔ bộ `(i, j, x)` với `i ≠ j` (màu đoạn đầu và đoạn sau), `1 ≤ x ≤ n − 1`, `x ≤ limit[i]`, `n − x ≤ limit[j]`. Hai bộ khác nhau cho hai cách sơn khác nhau (khác màu ở ít nhất một tờ), vì hai màu khác nhau và điểm cắt xác định cách tô.
2. **Đếm `x`:** điều kiện ⇔ `x ∈ [max(1, n − limit[j]), min(n − 1, limit[i])]` = `[n − A_j, A_i]` (vì `A_j ≤ n − 1` ⇒ `n − A_j ≥ 1`). Số phần tử `max(0, A_i + A_j − (n − 1))`.
3. **Đối xứng:** công thức đối xứng theo `i, j` ⇒ tổng có thứ tự = `2 ×` tổng không thứ tự.
4. **Tổng nhanh:** với `A` đã sắp và `i` cố định, tập `{j > i : A_i + A_j > c}` là hậu tố `[p, m)`; công thức cộng tách thành phần đếm và tổng hậu tố.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m² · n)` | `O(1)` |
| 2. Mọi cặp + công thức | `O(m²)` | `O(m)` |
| 3. Sort + bisect + tiền tố | `O(m log m)` | `O(m)` |
| 4. Sort + hai con trỏ | `O(m log m)` | `O(m)` |

Giá trị trước modulo có thể tới `m² · n ≈ 10^{19}` — Python xử lý được; ngôn ngữ khác phải lấy modulo từng bước.

## 12. Bộ test

In [13]:
def run_all(n, limit, with_brute=True):
    res = [pairs_quadratic(n, limit), sort_bisect(n, limit), two_pointers(n, limit), Solution().numberOfWays(n, limit)]
    if with_brute:
        res.append(brute(n, limit))
    return res

for n, lim, exp in examples:
    assert all(v == exp for v in run_all(n, lim))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    (2, [1, 1], 2),                  # n = 2: x = 1, hai thứ tự
    (2, [5, 5, 5], 6),               # 3·2 cặp có thứ tự, mỗi cặp 1 cách
    (10, [1, 1], 0),                 # tổng giới hạn quá nhỏ
    (5, [4, 1], 2),                  # (0,1): x=4 ; (1,0): x=1
    (6, [100, 100], 10),             # mỗi chiều 5 cách
]
for n, lim, exp in cases:
    r = run_all(n, lim)
    assert all(v == exp for v in r), (n, lim, r)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 phản ví dụ: quên cắt limit về n - 1
def wrong_no_cap(n, limit):
    c = n - 1; A = list(limit); tot = 0
    for i in range(len(A)):
        for j in range(i + 1, len(A)):
            tot += max(0, A[i] + A[j] - c)
    return 2 * tot % MOD

print("sai:", wrong_no_cap(6, [100, 100]), "| đúng:", Solution().numberOfWays(6, [100, 100]))

sai: 390 | đúng: 10


In [16]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3802)
for _ in range(300):
    n = rng.randint(2, 15); m = rng.randint(2, 6)
    lim = [rng.randint(1, 20) for _ in range(m)]
    r = run_all(n, lim)
    assert len(set(r)) == 1, (n, lim, r)
print("300 test nhỏ (có vét cạn): OK")
for _ in range(50):
    n = rng.randint(2, 10**9); lim = [rng.randint(1, 10**9) for _ in range(rng.randint(2, 300))]
    r = run_all(n, lim, with_brute=False)
    assert len(set(r)) == 1
print("50 test lớn giá trị (so với O(m^2)): OK")

300 test nhỏ (có vét cạn): OK
50 test lớn giá trị (so với O(m^2)): OK


In [17]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
lim = [rng.randint(1, 10**9) for _ in range(10**5)]
t0 = time.time(); a = Solution().numberOfWays(10**9, lim); t1 = time.time()
print("đáp án:", a, "| khớp hai con trỏ:", a == two_pointers(10**9, lim), "| thời gian < 1s:", t1 - t0 < 1)

đáp án: 825382082 | khớp hai con trỏ: True | thời gian < 1s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không cắt `limit` về `n − 1`

Đã thấy ở 12.3: với `limit` rất lớn, công thức `A_i + A_j − (n − 1)` vượt quá số điểm cắt thực tế `n − 1`. **Sửa:** `A_i = min(limit[i], n − 1)`.

### Lỗi 2 — Quên nhân 2 (đếm cặp không thứ tự)

Đổi thứ tự hai màu cho cách sơn **khác** (tờ đầu khác màu).

In [18]:
print("không nhân 2:", Solution().numberOfWays(3, [2, 2]) // 2, "| đúng:", Solution().numberOfWays(3, [2, 2]))

không nhân 2: 2 | đúng: 4


### Lỗi 3 — Cho phép `i = j` (cùng một màu)

Đề yêu cầu **đúng hai** màu khác nhau. Nếu tính cả `i = j` ta đếm thêm các cách "một màu cắt đôi" — không hợp lệ.

In [19]:
def wrong_same_color(n, limit):
    c = n - 1; A = [min(x, c) for x in limit]
    return sum(max(0, a + b - c) for a in A for b in A) % MOD     # SAI: gồm i == j

print("sai:", wrong_same_color(3, [2, 2]), "| đúng:", Solution().numberOfWays(3, [2, 2]))

sai: 8 | đúng: 4


### Lỗi 4 — Tìm nhị phân ngưỡng sai (`≥ c − A_i` thay vì `> c − A_i`)

Cặp chỉ đóng góp khi `A_i + A_j > c` (đóng góp `0` khi bằng — vô hại), nhưng nếu dùng ngưỡng `c − A_i` (gồm cả trường hợp tổng **nhỏ hơn**…) thì sai. Kiểm tra lại: với `bisect_left(A, c − x)`, các `j` có `A_j = c − x` đóng góp `0` nên vẫn đúng; với `bisect_left(A, c − x − 1)` thì đóng góp **âm** lọt vào.

In [20]:
def wrong_threshold(n, limit):
    c = n - 1; A = sorted(min(x, c) for x in limit); m = len(A)
    pre = [0] * (m + 1)
    for i, x in enumerate(A): pre[i + 1] = pre[i] + x
    tot = 0
    for i, x in enumerate(A):
        p = max(i + 1, bisect_left(A, c - x - 1))                  # SAI: lấy cả cặp có tổng = c - 1
        if p < m: tot += (m - p) * (x - c) + pre[m] - pre[p]
    return 2 * tot % MOD

print("sai:", wrong_threshold(5, [1, 3, 2, 2]), "| đúng:", Solution().numberOfWays(5, [1, 3, 2, 2]), "| vét cạn:", brute(5, [1, 3, 2, 2]))

sai: 0 | đúng: 4 | vét cạn: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đếm cấu hình gồm "cặp đối tượng + tham số liên tục"** ⇒ đếm tham số bằng công thức giao khoảng, rồi lo phần cặp.
- **`Σ_{i<j} max(0, a_i + a_j − c)`** ⇒ sắp xếp + hậu tố (bisect hoặc hai con trỏ) + tổng tiền tố — mẫu "đếm cặp có tổng ≥ ngưỡng" (LC 2563 *Count the Number of Fair Pairs*).
- Công thức **đối xứng** ⇒ nhân 2 thay vì đếm có thứ tự.
- Luôn **cắt** tham số về miền hợp lệ (`≤ n − 1`) trước khi dùng công thức tuyến tính.

## 15. Biến thể của bài

1. **Mỗi màu có cả giới hạn dưới** `low[i]`: số `x` = độ dài giao `[max(low_i, n − hi_j), min(hi_i, n − low_j)]` — không còn dạng `A_i + A_j − c` đơn giản.
2. **Đúng ba màu** (ba đoạn): cố định màu giữa, đếm hai điểm cắt — tổng hai chiều.
3. **Màu có thể lặp lại không liền kề** / nhiều đoạn: DP.
4. **Không cần liên tiếp** (mỗi tờ chọn một trong hai màu): tổ hợp `C(n, x)` thay cho 1.

In [21]:
# Biến thể 2: đúng ba màu khác nhau, ba đoạn liên tiếp (vét cạn để minh họa với dữ liệu nhỏ)
def three_colors(n, limit):
    m = len(limit); cnt = 0
    for a in range(m):
        for b in range(m):
            for c in range(m):
                if len({a, b, c}) < 3: continue
                for x in range(1, n - 1):
                    for y in range(1, n - x):
                        z = n - x - y
                        if x <= limit[a] and y <= limit[b] and 1 <= z <= limit[c]:
                            cnt += 1
    return cnt

print(three_colors(4, [2, 2, 2]), three_colors(5, [3, 1, 2, 2]))

18 54


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi cặp, mọi `x` | `O(m² n)` | `O(1)` | Kiểm thử |
| 2. Mọi cặp + công thức | `max(0, A_i + A_j − c)` | `O(m²)` | `O(m)` | `m ≤ 3000` |
| 3. Sort + bisect + tiền tố | Hậu tố hợp lệ | `O(m log m)` | `O(m)` | **Lời giải chuẩn** |
| 4. Sort + hai con trỏ | Con trỏ đơn điệu | `O(m log m)` | `O(m)` | Tránh `bisect` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `numberOfWays(5, [2, 2])` = ?

**Bài 2 (tính tay).** `numberOfWays(5, [4, 3, 1])`.

**Bài 3 (điền TODO).** Viết `count_pairs_sum_gt(a, c)` = số cặp `i < j` có `a_i + a_j > c` (sắp xếp + hai con trỏ).

**Bài 4 (tự viết lại).** Viết lời giải lấy modulo **từng bước** (mô phỏng ngôn ngữ có giới hạn số nguyên) và đối chiếu.

**Bài 5 (mini challenge).** Biến thể có giới hạn dưới `low[i]` (mỗi màu phải sơn ít nhất `low[i]` tờ). Viết lời giải `O(m²)` và kiểm chứng.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_pairs_sum_gt(a, c):
    # TODO: sắp a; hai con trỏ lo, hi
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def with_lower_bounds(n, low, high):
    # TODO: với mỗi cặp (i, j): x trong [max(low_i, n - high_j, 1), min(high_i, n - low_j, n - 1)]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `A = [2, 2]`, `c = 4`. (2) `2 + 2 − 4 = 0`. (3) Đáp án `0` (cần ít nhất 5 tờ nhưng tổng giới hạn chỉ 4).

**Bài 2.** (1) `A = [4, 3, 1]`, `c = 4`. (2) Cặp `(4,3)`: 3; `(4,1)`: 1; `(3,1)`: 0. (3) `2·4 = 8`.

**Bài 3.** (1) `lo = 0, hi = len − 1`. (2) Nếu `a[lo] + a[hi] > c` thì mọi `lo' ∈ [lo, hi)` ghép với `hi` đều được ⇒ cộng `hi − lo`, giảm `hi`. (3) Ngược lại tăng `lo`.

**Bài 4.** (1) `% MOD` sau mỗi phép cộng/nhân. (2) Cẩn thận `(x − c)` âm: dùng `(x − c) % MOD`. (3) Đối chiếu.

**Bài 5.** (1) Giao khoảng như mục 3.1 với thêm cận dưới. (2) `O(m²)` cặp. (3) Vét cạn `x`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().numberOfWays(5, [2, 2]) == 0 == brute(5, [2, 2])
assert Solution().numberOfWays(5, [4, 3, 1]) == 8 == brute(5, [4, 3, 1])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3
def count_pairs_sum_gt_sol(a, c):
    a = sorted(a); lo, hi, cnt = 0, len(a) - 1, 0
    while lo < hi:
        if a[lo] + a[hi] > c:
            cnt += hi - lo; hi -= 1
        else:
            lo += 1
    return cnt

rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(0, 20) for _ in range(rng.randint(0, 10))]; c = rng.randint(0, 30)
    assert count_pairs_sum_gt_sol(a, c) == sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] + a[j] > c)
print("Bài 3 OK")

Bài 3 OK


In [26]:
# Bài 4: modulo từng bước
def step_mod(n, limit):
    c = n - 1; A = sorted(min(x, c) for x in limit); m = len(A)
    pre = [0] * (m + 1)
    for i, x in enumerate(A): pre[i + 1] = (pre[i] + x) % MOD
    tot = 0
    for i, x in enumerate(A):
        p = max(i + 1, bisect_left(A, c - x + 1))
        if p < m:
            tot = (tot + (m - p) * ((x - c) % MOD) + pre[m] - pre[p]) % MOD
    return 2 * tot % MOD

rng = random.Random(4)
for _ in range(100):
    n = rng.randint(2, 10**9); lim = [rng.randint(1, 10**9) for _ in range(rng.randint(2, 200))]
    assert step_mod(n, lim) == Solution().numberOfWays(n, lim)
print("Bài 4 OK")

Bài 4 OK


In [27]:
# Bài 5: có giới hạn dưới
def with_lower_bounds_sol(n, low, high):
    m = len(low); tot = 0
    for i in range(m):
        for j in range(m):
            if i == j: continue
            lo = max(low[i], n - high[j], 1); hi = min(high[i], n - low[j], n - 1)
            tot += max(0, hi - lo + 1)
    return tot % MOD

def brute_low(n, low, high):
    m = len(low)
    return sum(1 for i in range(m) for j in range(m) if i != j for x in range(1, n)
               if low[i] <= x <= high[i] and low[j] <= n - x <= high[j]) % MOD

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(2, 12); m = rng.randint(2, 5)
    low = [rng.randint(1, 5) for _ in range(m)]; high = [l + rng.randint(0, 6) for l in low]
    assert with_lower_bounds_sol(n, low, high) == brute_low(n, low, high)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
c = n - 1
A = sorted(min(x, c) for x in limit)
pre = tiền tố của A
tot = 0
for i, x in enumerate(A):
    p = max(i + 1, bisect_left(A, c - x + 1))     # j >= p  <=>  A_i + A_j > c
    if p < m: tot += (m - p) * (x - c) + pre[m] - pre[p]
return 2 * tot % MOD
```

**Checklist:**

- [ ] Cắt `limit` về `n − 1`?
- [ ] Nhân 2 cho hai thứ tự màu?
- [ ] Loại `i = j`?
- [ ] Ngưỡng tìm nhị phân `c − A_i + 1` (tổng > c)?
- [ ] Lấy modulo (từng bước nếu ngôn ngữ có giới hạn số nguyên)?
- [ ] (Premium) Đã đối chiếu tên phương thức `numberOfWays`?